# Quick Verification – Metaheuristic Designer
This notebook runs a tiny (20+30)‑ES on the MaxOnes problem to check that all refactored components work together.

In [1]:
import numpy as np
import logging
import metaheuristic_designer as mhd

# For reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

from metaheuristic_designer.operators import create_operator
from metaheuristic_designer.operators.mutation_operator import create_mutation_operator
from metaheuristic_designer.operators.crossover_operator import create_crossover_operator
from metaheuristic_designer.parent_selection_methods import create_parent_selection
from metaheuristic_designer.survivor_selection_methods import create_survivor_selection
from metaheuristic_designer.initializers import UniformInitializer
from metaheuristic_designer.encoding import DefaultEncoding
from metaheuristic_designer.strategies import ES, GA
from metaheuristic_designer.algorithms import StandardAlgorithm

In [2]:
# Get the root logger for your package
# mhd_logger = logging.getLogger("metaheuristic_designer")
mhd_logger = logging.getLogger("metaheuristic_designer.parametrizable_mixin")

# Set the level you want:
mhd_logger.setLevel(logging.DEBUG)   # or logging.INFO

# Ensure messages actually appear (if no handler is configured yet)
if not mhd_logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(asctime)s [%(name)s] %(levelname)s: %(message)s')
    handler.setFormatter(formatter)
    mhd_logger.addHandler(handler)

### 1. Define a trivial problem

In [3]:
class MaxOnes(mhd.VectorObjectiveFunc):
    def __init__(self, vecsize):
        super().__init__(vecsize=vecsize, low_lim=0, up_lim=1, mode="max", name="MaxOnes")

    def objective(self, x):
        return x.sum()

vecsize = 50
objfunc = MaxOnes(vecsize)

2026-04-26 21:58:53,306 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for MaxOnes named "MaxOnes".


### 2. Parameters

In [4]:
pop_size = 20
offspring_size = 30

### 3. Build operators (factory)

In [5]:
mutation = create_operator(
    "mutation.bernoulli", loc=0.0, scale=0.2, N=1, random_state=rng
)
crossover = create_operator("crossover.1point")

2026-04-26 21:58:53,312 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for DefaultEncoding named "unknown".
2026-04-26 21:58:53,313 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for OperatorFromLambda named "mutation.bernoulli".
2026-04-26 21:58:53,313 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for DefaultEncoding named "unknown".
2026-04-26 21:58:53,313 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for DefaultEncoding named "unknown".
2026-04-26 21:58:53,313 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for OperatorFromLambda named "crossover.1point".
2026-04-26 21:58:53,314 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for DefaultEncoding named "unknown".


### 4. Build selection methods (factory)

In [6]:
parent_sel = create_parent_selection("random", amount=offspring_size, random_state=rng)
survivor_sel = create_survivor_selection("(m+n)", random_state=rng)

2026-04-26 21:58:53,316 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for ParentSelectionFromLambda named "random".
2026-04-26 21:58:53,317 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for SurvivorSelectionFromLambda named "(m+n)".


### 5. Initializer & Strategy

In [7]:
initializer = UniformInitializer(
    vecsize, 0, 1, pop_size=pop_size,
    dtype=float, encoding=DefaultEncoding()
    , random_state=rng
)

strategy = ES(
    initializer=initializer,
    mutation_op=mutation,
    cross_op=crossover,
    parent_sel=parent_sel,
    survivor_sel=survivor_sel,
    params={"offspringSize": offspring_size}
)

2026-04-26 21:58:53,320 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for DefaultEncoding named "unknown".
2026-04-26 21:58:53,320 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for DefaultEncoding named "unknown".
2026-04-26 21:58:53,321 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for CompositeOperator named "Sequence (mutation.bernoulli, crossover.1point)".
2026-04-26 21:58:53,321 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for DefaultEncoding named "unknown".
2026-04-26 21:58:53,321 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for ES named "ES".
2026-04-26 21:58:53,321 [metaheuristic_designer.parametrizable_mixin] DEBUG: Evaluated every parameter at 0.0000 for CompositeOperator named "Sequence (mutation.bernoulli, crossover.1point)".
2026-04-26 21:58:53,321 [metaheuri

### 6. Algorithm & Run

In [8]:
alg = StandardAlgorithm(
    objfunc, strategy,
    **{"stop_cond": "ngen", "ngen": 50, "verbose": True, "v_timer": -1}
)

print("Starting optimization…")
final_pop = alg.optimize()

TypeError: StoppingCondition.__init__() got an unexpected keyword argument 'condition_str'

In [ ]:
best_sol, best_fit = alg.best_solution()
print(f"\nBest fitness: {best_fit} / {vecsize} (ideal)")
print("First 10 genes:", best_sol[:10].astype(int))

If everything is wired correctly, you should see the fitness climb toward 50. The exact final value depends on the random seed, but it should be > 40 after 50 generations.

In [ ]:
#!/usr/bin/env python3
"""
Esoteric integration test – Scheduled ES with composite operator,
TypeCastEncoding, and conditional elitism on an integer-matching problem.
"""

import numpy as np
from metaheuristic_designer import VectorObjectiveFunc
from metaheuristic_designer.algorithms import StandardAlgorithm
from metaheuristic_designer.initializers import UniformInitializer
from metaheuristic_designer.encodings import TypeCastEncoding
from metaheuristic_designer.operators import create_operator
from metaheuristic_designer.operators.composite_operator import CompositeOperator
from metaheuristic_designer.parent_selection_methods import create_parent_selection
from metaheuristic_designer.survivor_selection_methods import create_survivor_selection
from metaheuristic_designer.strategies import ES
from metaheuristic_designer.parameter_schedules import LinearSchedule  # if you have this

# ----- 1. Objective: minimise absolute difference to a random target vector -----
vecsize = 20
target = np.random.default_rng(999).integers(0, 10, size=vecsize)  # small integers

class MatchTarget(VectorObjectiveFunc):
    def __init__(self, target):
        self.target = target
        super().__init__(vecsize=len(target), low_lim=0, up_lim=9,
                         mode="min", name="MatchTarget", vectorized=True)

    def objective(self, x):
        return np.sum(np.abs(x - self.target), axis=1)   # vectorized, x is 2D

objfunc = MatchTarget(target)

# ----- 2. Encoding: internal continuous, decode to integer -----
encoding = TypeCastEncoding(encoded_dtype=float, decoded_dtype=int)

# ----- 3. Parameter scheduling for mutation scale (linear decay) -----
# Use a simple lambda; the mixin’s store_kwargs accepts callables
scale_schedule = lambda p: 1.0 - 0.9 * p   # from 1.0 down to 0.1

# ----- 4. Build operators via generic factory (dot notation) -----
crossover = create_operator("crossover.1point", random_state=42)
mutation = create_operator("mutation.gauss",
                           loc=0.0,
                           scale=scale_schedule,   # <-- scheduled parameter
                           N=1,
                           random_state=42)
# Compose: crossover then mutation
composite_op = CompositeOperator([crossover, mutation], name="Xover+Mut")

# ----- 5. Selection methods (also via factories) -----
parent_sel = create_parent_selection("tournament",
                                     amount=20,       # offspring = pop_size
                                     tournament_size=3,
                                     random_state=42)
survivor_sel = create_survivor_selection("cond_elitism",
                                         amount=2,     # keep 2 elites
                                         random_state=42)

# ----- 6. Strategy & algorithm -----
initializer = UniformInitializer(vecsize, 0, 9, pop_size=10, dtype=float, encoding=encoding)
strategy = GA(initializer=initializer,
              mutation_op=composite_op,   # note: we give composite directly
              cross_op=None,              # already inside composite
              parent_sel=parent_sel,
              survivor_sel=survivor_sel,
              params={"offspringSize": 20})

alg = StandardAlgorithm(objfunc, strategy,
                       params={"stop_cond": "ngen", "ngen": 30,
                               "verbose": True, "v_timer": 0.5})

# ----- 7. Run -----
print("\nRunning esoteric ES with scheduling, composite op, integer encoding...")
final_pop = alg.optimize()
best_sol, best_fit = alg.best_solution(decoded=True)
print(f"\nBest fitness (abs error): {best_fit}")
print("Decoded best solution (should be integers):", best_sol.astype(int))

# The error should be well below the initial random expectation (~22.5 per dimension? Actually for 20 dimensions, random uniform 0-9, expected error ~45? initial fitness usually > 100, we expect < 20 after 30 gens)
assert best_fit < 30, f"Fitness {best_fit} still high, algorithm may not be improving."
print("✓ Esoteric test passed — scheduling, composite op, encoding, and selection all work together.")